# 🤖 LSTM Stock Prediction Model Training

## Indonesian Stock Prediction with Deep Learning

**Free GPU Training on Google Colab**

---

### Setup Instructions:
1. Click **Runtime → Change runtime type**
2. Select **GPU** (T4 GPU is free!)
3. Click **Save**
4. Run all cells sequentially

**Training Time:** ~5-10 minutes with GPU

---

In [ ]:
# Step 1: Clone Repository
print("📥 Cloning repository...")
!git clone https://github.com/herrylim2001/stock-prediction-indonesia.git
%cd stock-prediction-indonesia
!git checkout claude/review-repo-files-fdrwt
print("✓ Repository cloned!")

In [ ]:
# Step 2: Install Dependencies
print("📦 Installing dependencies...")
!pip install -q yfinance pandas numpy ta scikit-learn joblib
print("✓ Dependencies installed!")

# Check GPU
import tensorflow as tf
print(f"\n🎮 GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Step 3: Quick Test - Fetch Sample Data
print("🧪 Testing data fetching...\n")

import sys
sys.path.append('/content/stock-prediction-indonesia')

from models.data_collector import StockDataCollector

collector = StockDataCollector()
test_df = collector.fetch_historical_data('BBCA', period='3mo')

if test_df is not None:
    print(f"✓ Successfully fetched {len(test_df)} rows for BBCA")
    print(f"  Date range: {test_df['date'].min()} to {test_df['date'].max()}")
else:
    print("⚠️ Data fetch failed - will retry during training")

In [ ]:
# Step 4: Start Training (Default: 6 stocks, 50 epochs)
print("="*70)
print("🚀 STARTING MODEL TRAINING")
print("="*70)
print("\nTraining Configuration:")
print("  Stocks: BBCA, BBRI, TLKM, ASII, BMRI, UNVR")
print("  Period: 2 years")
print("  Epochs: 50 (faster for Colab)")
print("  GPU: Enabled")
print("\n" + "="*70 + "\n")

# Run training
!python models/train_model.py \
  --stocks BBCA BBRI TLKM ASII BMRI UNVR \
  --period 2y \
  --epochs 50 \
  --batch-size 32

In [ ]:
# Step 5: Test Trained Model
print("\n" + "="*70)
print("🧪 TESTING TRAINED MODEL")
print("="*70 + "\n")

!python models/train_model.py --test

In [ ]:
# Step 6: Check Model Files
import os

print("\n📁 Checking saved model files...\n")

model_files = [
    'models/saved/stock_lstm_model.keras',
    'models/saved/training_metadata.pkl',
    'models/checkpoints/lstm_best.keras'
]

for filepath in model_files:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"✓ {filepath} ({size_mb:.2f} MB)")
    else:
        print(f"❌ {filepath} - NOT FOUND")

print("\n" + "="*70)

In [ ]:
# Step 7: Make Sample Prediction
print("\n🎯 Making sample prediction...\n")

from models.predictor import StockPredictor
from models.data_collector import StockDataCollector

# Initialize predictor
predictor = StockPredictor(
    model_path='models/saved/stock_lstm_model.keras',
    metadata_path='models/saved/training_metadata.pkl'
)

if predictor.is_model_loaded():
    print("✓ Model loaded successfully!\n")
    
    # Fetch recent data for BBCA
    collector = StockDataCollector()
    df = collector.fetch_historical_data('BBCA', period='3mo')
    df = collector.add_technical_indicators(df)
    
    current_price = df['close'].iloc[-1]
    
    # Make prediction
    predictions = predictor.predict_multiple_horizons(df, 'BBCA', current_price)
    
    print("\n📊 Sample Prediction for BBCA:")
    print(f"Current Price: Rp {current_price:,.0f}\n")
    
    for horizon, pred in predictions.items():
        change_pct = ((pred['price'] - current_price) / current_price) * 100
        print(f"{horizon.upper()}:")
        print(f"  Predicted: Rp {pred['price']:,.0f} ({change_pct:+.2f}%)")
        print(f"  Trend: {pred['trend']}")
        print(f"  Confidence: {pred['confidence']:.1%}")
        print(f"  Model: {pred.get('model_used', 'LSTM')}")
        print()
else:
    print("❌ Model not loaded - check training output above")

In [ ]:
# Step 8: Download Trained Model Files
print("\n💾 Preparing model files for download...\n")

# Create zip file with all model files
!zip -r trained_model.zip models/saved/ models/checkpoints/

print("\n✓ Model files zipped!")
print("\nTo download:")
print("1. Click the folder icon on the left sidebar")
print("2. Find 'trained_model.zip'")
print("3. Right-click → Download")
print("\nOr run the cell below to download directly:")

In [ ]:
# Auto-download (Google Colab only)
from google.colab import files
files.download('trained_model.zip')
print("✓ Download started!")

---

## 🎉 Training Complete!

### Next Steps:

1. **Download** `trained_model.zip` (previous cell)
2. **Extract** the zip file
3. **Upload** to your GitHub repository:
   ```bash
   # On your local machine
   unzip trained_model.zip
   git add models/saved/
   git commit -m "Add trained LSTM model"
   git push
   ```

4. **Deploy** to Streamlit Cloud
   - Streamlit will automatically detect the model files
   - Dashboard will switch from mock to real LSTM predictions!

### Model Performance:
- Check the training output above for final metrics
- Target: MAE < 10%, MAPE < 10%
- Expected accuracy: 65-75% direction prediction

### Troubleshooting:
- If training fails, try with fewer stocks: `--stocks BBCA BBRI`
- If out of memory, reduce batch size: `--batch-size 16`
- For faster testing, use fewer epochs: `--epochs 20`

---

**Need help?** Check `LSTM_TRAINING_GUIDE.md` in the repository!
